# 06 — Topic Modeling
**Goal:** Descubrir automáticamente los temas que aparecen en un corpus sin etiquetas.

## Latent Dirichlet Allocation (LDA)

LDA es un modelo generativo: asume que cada documento es una mezcla de temas, y cada tema es una distribución de palabras.

```
doc → [30% App, 50% OTP, 20% Proceso]     ← distribución de temas
tema_App  → {app: 0.3, lenta: 0.2, ...}   ← distribución de palabras
```

Referencia: Blei, Ng & Jordan (2003). sklearn usa Variational Bayes.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
})

STOPWORDS_ES = [
    'a','al','algo','ante','como','con','de','del','desde','el','ella','en',
    'entre','es','esta','este','esto','fue','la','las','le','lo','los','me',
    'mi','muy','no','nos','o','para','pero','por','que','se','si','sin',
    'su','sus','también','te','todo','un','una','y','ya','yo','tuve','llevo',
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
# Corpus mixto: varios temas de soporte fintech
topic_templates = {
    'App / UX': [
        'La app está muy lenta y tarda en cargar la pantalla',
        'La aplicación no carga, sigue dando error al abrir',
        'La app se cierra sola cuando intento iniciar sesión',
        'Muy lenta la aplicación, tarda demasiado en responder',
        'La interfaz de la app es confusa y difícil de usar',
    ],
    'OTP / Seguridad': [
        'No me llegó el OTP al celular para confirmar',
        'El código OTP no llega por SMS, ya intenté varias veces',
        'El código de verificación no aparece en mi teléfono',
        'Necesito el OTP pero no llega ningún mensaje al celular',
        'El SMS con el código tarda demasiado o no llega nunca',
    ],
    'Documentos': [
        'La carga de documentos falla constantemente en la app',
        'No puedo subir los documentos requeridos, siempre falla',
        'El proceso de carga de documentos da error todo el tiempo',
        'Los documentos no se suben correctamente al sistema',
        'Intenté cargar mi documento de identidad tres veces sin éxito',
    ],
    'Crédito / Evaluación': [
        'Llevo días esperando la evaluación crediticia sin respuesta',
        'Mi solicitud de crédito lleva una semana en evaluación',
        'No hay información sobre el estado de mi evaluación crediticia',
        'Quiero saber cuándo aprueban mi solicitud de tarjeta',
        'La evaluación de crédito no termina nunca, mucho tiempo',
    ],
    'Soporte / Atención': [
        'El call center no responde, llevo horas esperando',
        'Nadie del soporte responde mis mensajes ni llamadas',
        'Pésima atención al cliente, nadie resuelve mis problemas',
        'Llamé cuatro veces y siguen sin resolver mi caso',
        'El chat de soporte no funciona y el teléfono no contesta',
    ],
    'Experiencia positiva': [
        'Excelente servicio, aprobaron mi tarjeta muy rápido',
        'El proceso de solicitud fue fácil y muy eficiente',
        'Increíbles beneficios, recomiendo la tarjeta sin dudarlo',
        'Muy contento con el servicio, todo salió perfecto',
        'La activación fue sencilla y el proceso muy rápido',
    ],
}

rng = np.random.default_rng(42)
corpus = []
true_topics = []
for topic, templates in topic_templates.items():
    for _ in range(100):
        corpus.append(templates[rng.integers(len(templates))])
        true_topics.append(topic)

df = pd.DataFrame({'text': corpus, 'true_topic': true_topics})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Corpus: {len(df)} documentos | {df.true_topic.nunique()} temas reales')

## 1. Vectorizar con CountVectorizer (LDA necesita conteos, no TF-IDF)

In [ ]:
vec = CountVectorizer(
    preprocessor=preprocess,
    stop_words=STOPWORDS_ES,
    min_df=3,
    max_df=0.90,
    ngram_range=(1, 1),   # LDA funciona mejor con unigramas
)

X = vec.fit_transform(df['text'])
feature_names = vec.get_feature_names_out()
print(f'DTM shape: {X.shape}  |  vocab: {len(feature_names)} tokens')

## 2. Entrenar LDA

In [ ]:
N_TOPICS = 6   # sabemos que hay 6 temas reales

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    max_iter=20,
    learning_method='batch',
    random_state=42,
    doc_topic_prior=0.1,    # α: docs focalizados en pocos temas
    topic_word_prior=0.01,  # β: temas con vocabulario específico
)

lda.fit(X)
print(f'Perplexity (menor es mejor): {lda.perplexity(X):.1f}')

In [ ]:
# Top palabras por tema
def print_topics(lda, feature_names, top_n=10):
    for topic_idx, topic in enumerate(lda.components_):
        top_words_idx = topic.argsort()[::-1][:top_n]
        top_words = [feature_names[i] for i in top_words_idx]
        print(f'Tema {topic_idx}: {" | ".join(top_words)}')

print_topics(lda, feature_names)

## 3. Visualizar distribución de temas

In [ ]:
# Distribución de temas por documento
doc_topics = lda.transform(X)   # (600, 6) — probabilidades por tema

# Tema dominante de cada doc
df['lda_topic'] = doc_topics.argmax(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribución de temas LDA
topic_counts = df['lda_topic'].value_counts().sort_index()
axes[0].bar(range(N_TOPICS), topic_counts.values, color='#4361ee')
axes[0].set_xticks(range(N_TOPICS))
axes[0].set_xticklabels([f'Tema {i}' for i in range(N_TOPICS)])
axes[0].set_title('Docs por tema LDA (tema dominante)')
axes[0].set_ylabel('N documentos')

# Distribución de temas reales
real_counts = df['true_topic'].value_counts()
axes[1].barh(real_counts.index, real_counts.values, color='#f72585')
axes[1].set_title('Docs por tema REAL')
axes[1].set_xlabel('N documentos')

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: tema LDA vs tema real
pivot = pd.crosstab(df['true_topic'], df['lda_topic'])

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(pivot.values, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(N_TOPICS))
ax.set_xticklabels([f'LDA-{i}' for i in range(N_TOPICS)])
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_title('Tema real vs tema LDA asignado')
for i in range(len(pivot)):
    for j in range(N_TOPICS):
        ax.text(j, i, pivot.values[i, j], ha='center', va='center',
                fontsize=9, color='white' if pivot.values[i, j] > 40 else 'black')
plt.tight_layout()
plt.show()

## 4. Encontrar el número óptimo de temas — perplexity

In [ ]:
perplexities = []
ks = range(2, 12)

for k in ks:
    m = LatentDirichletAllocation(n_components=k, max_iter=15,
                                   learning_method='batch', random_state=42)
    m.fit(X)
    perplexities.append(m.perplexity(X))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(ks), perplexities, 'o-', color='#4361ee', linewidth=2)
ax.axvline(6, color='#f72585', linestyle='--', label='k=6 (real)')
ax.set_xlabel('Número de temas (k)')
ax.set_ylabel('Perplexity')
ax.set_title('Perplexity vs número de temas LDA\n(buscar codo o valor real esperado)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Uso práctico — etiquetar nuevos documentos

In [ ]:
# Etiquetas manuales que le asignamos a cada tema LDA
# (mirando las top palabras y ajustando)
TOPIC_LABELS = {
    0: 'Tema A', 1: 'Tema B', 2: 'Tema C',
    3: 'Tema D', 4: 'Tema E', 5: 'Tema F',
}

nuevos_tickets = [
    'La aplicación no abre y tarda mucho',
    'No recibo el código SMS para verificar',
    'Cuándo aprueban mi solicitud de crédito',
    'El soporte no me responde hace dos días',
    'Muy contento con el servicio, todo excelente',
]

X_new = vec.transform(nuevos_tickets)
topic_probs = lda.transform(X_new)  # (5, 6)

print(f'{"Ticket":<50} {"Tema dominante":>15} {"Prob":>6}')
print('-' * 75)
for ticket, probs in zip(nuevos_tickets, topic_probs):
    top_topic = probs.argmax()
    print(f'{ticket:<50} {TOPIC_LABELS[top_topic]:>15} {probs[top_topic]:>6.3f}')

## Resumen
| Concepto | Detalle |
|---|---|
| LDA | Modelo generativo: docs = mezcla de temas; temas = distribución de palabras |
| `CountVectorizer` | LDA necesita conteos enteros, no TF-IDF |
| `lda.components_` | Matriz (n_topics, vocab) — distribución de palabras por tema |
| `lda.transform(X)` | Distribución de temas por documento |
| `lda.perplexity(X)` | Métrica de ajuste — menor es mejor |
| `doc_topic_prior` α | Menor → docs más focalizados en pocos temas |
| `topic_word_prior` β | Menor → temas con vocabulario más específico |

**Siguiente:** `07_text_classification.ipynb` — pipeline completo de clasificación de texto con sklearn.